# UNSW-NB15 IDS — Phase 1 Artifact Preparation

This notebook is a **separate, self-contained artifact-preparation notebook** for the Phase 1 → Phase 2 handoff.

It is designed to solve the missing-variable problem encountered when moving from the individual-model notebook to the ensemble notebook.

### What this notebook does

1. Loads the official UNSW-NB15 training and testing CSV files.
2. Recreates the same preprocessing used by the Phase 1 pipeline.
3. Uses the exact 19 selected features.
4. Creates `X_test_selected` and `y_test`.
5. Loads existing Phase-1 fitted models and OOF predictions **if they were previously saved**.
6. If OOF predictions are unavailable, it provides a clearly marked option to regenerate them from the existing tuned models.
7. Saves all artifacts required by the ensemble notebook.

### Important

This notebook **does not perform hyperparameter tuning**.

The goal is to prepare a clean handoff between Phase 1 and Phase 2 without rerunning expensive model tuning.

In [1]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

print("Setup complete.")

Setup complete.


## 1. File paths

Place these files in the same working directory as this notebook:

- `UNSW_NB15_training-set.csv`
- `UNSW_NB15_testing-set.csv`

The notebook also uses:

```text
phase1_artifacts/
```

for saved Phase-1 objects.

In [2]:
TRAIN_PATH = "UNSW_NB15_training-set.csv"
TEST_PATH = "UNSW_NB15_testing-set.csv"

PHASE1_ARTIFACT_DIR = "phase1_artifacts"
os.makedirs(PHASE1_ARTIFACT_DIR, exist_ok=True)

print("Working directory:", os.getcwd())
print("Train exists:", os.path.exists(TRAIN_PATH))
print("Test exists :", os.path.exists(TEST_PATH))

Working directory: C:\Users\aksha\OneDrive\Pictures\Documents\Dummy
Train exists: True
Test exists : True


In [3]:
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(
        f"Training file not found: {TRAIN_PATH}\n"
        "Put UNSW_NB15_training-set.csv in the notebook's working directory."
    )

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(
        f"Testing file not found: {TEST_PATH}\n"
        "Put UNSW_NB15_testing-set.csv in the notebook's working directory."
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training shape:", train_df.shape)
print("Testing shape :", test_df.shape)

Training shape: (175341, 45)
Testing shape : (82332, 45)


## 2. Exact 19-feature definition

These are the selected features from the current Phase-1 pipeline.

Do not add or remove features here. The ensemble must use exactly the same feature representation as the individual models.

In [4]:
selected_features = [
    'ct_dst_src_ltm',
    'sttl',
    'sbytes',
    'smean',
    'dmean',
    'dttl',
    'dbytes',
    'ct_state_ttl',
    'service_none',
    'ct_dst_sport_ltm',
    'dload',
    'proto_udp',
    'sloss',
    'state_int',
    'proto_tcp',
    'service_http',
    'tcprtt',
    'swin',
    'response_body_len'
]

print("Number of selected features:", len(selected_features))
print(selected_features)

Number of selected features: 19
['ct_dst_src_ltm', 'sttl', 'sbytes', 'smean', 'dmean', 'dttl', 'dbytes', 'ct_state_ttl', 'service_none', 'ct_dst_sport_ltm', 'dload', 'proto_udp', 'sloss', 'state_int', 'proto_tcp', 'service_http', 'tcprtt', 'swin', 'response_body_len']


## 3. Recreate the Phase-1 preprocessing

The current Phase-1 pipeline:

- uses `label` as the binary target
- removes `id` and `attack_cat`
- normalizes the categorical columns `proto`, `service`, and `state`
- converts categorical variables with `pd.get_dummies(..., drop_first=False)`

The helper below is deliberately kept aligned with that workflow.

In [5]:
TARGET = "label"
CAT_COLS = ["proto", "service", "state"]

def prepare_dataframe(df):
    df_model = df.copy()

    if TARGET not in df_model.columns:
        raise ValueError(
            f"Target column '{TARGET}' is missing from the dataframe."
        )

    # Match Phase-1 cleaning
    df_model = df_model.drop(
        columns=["id", "attack_cat"],
        errors="ignore"
    )

    # Normalize categorical columns
    for col in CAT_COLS:
        if col in df_model.columns:
            df_model[col] = (
                df_model[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .replace("-", "none")
            )

    X = pd.get_dummies(
        df_model.drop(columns=[TARGET]),
        columns=[c for c in CAT_COLS if c in df_model.columns],
        drop_first=False
    )

    y = df_model[TARGET].astype(int)

    return X, y

## 4. Prepare training and official test data

The test set is processed independently, then its columns are aligned to the training representation.

This is important because one-hot encoding can produce slightly different columns between train and test.

In [6]:
X_train_all, y_train = prepare_dataframe(train_df)
X_test_all, y_test = prepare_dataframe(test_df)

print("Raw processed train shape:", X_train_all.shape)
print("Raw processed test shape :", X_test_all.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Raw processed train shape: (175341, 194)
Raw processed test shape : (82332, 190)
y_train: (175341,)
y_test : (82332,)


In [7]:
# Align test columns to training columns.
# Missing one-hot columns become zero.
X_test_all = X_test_all.reindex(
    columns=X_train_all.columns,
    fill_value=0
)

missing_selected_train = [
    f for f in selected_features
    if f not in X_train_all.columns
]

missing_selected_test = [
    f for f in selected_features
    if f not in X_test_all.columns
]

if missing_selected_train:
    raise ValueError(
        "Selected features missing from training data: "
        + str(missing_selected_train)
    )

if missing_selected_test:
    raise ValueError(
        "Selected features missing from test data after alignment: "
        + str(missing_selected_test)
    )

X_selected = X_train_all[selected_features].copy()
X_test_selected = X_test_all[selected_features].copy()

print("X_selected:", X_selected.shape)
print("X_test_selected:", X_test_selected.shape)

X_selected: (175341, 19)
X_test_selected: (82332, 19)


In [8]:
print("Training label distribution:")
display(
    y_train.value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print("Official test label distribution:")
display(
    y_test.value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print("Test positive rate:", round(y_test.mean(), 4))

Training label distribution:


,count
label,
0,56000
1,119341


Official test label distribution:


,count
label,
0,37000
1,45332


Test positive rate: 0.5506


## 5. Load existing Phase-1 artifacts

If you already saved:

```text
phase1_artifacts/oof_scores.joblib
phase1_artifacts/tuned_models.joblib
phase1_artifacts/y_train.joblib
```

they will be loaded automatically.

If they do not exist, the notebook will not pretend they exist. Instead, it will report exactly what is missing.

In [9]:
artifact_paths = {
    "oof_scores": os.path.join(PHASE1_ARTIFACT_DIR, "oof_scores.joblib"),
    "tuned_models": os.path.join(PHASE1_ARTIFACT_DIR, "tuned_models.joblib"),
    "saved_y_train": os.path.join(PHASE1_ARTIFACT_DIR, "y_train.joblib"),
}

loaded = {}

for name, path in artifact_paths.items():
    if os.path.exists(path):
        loaded[name] = joblib.load(path)
        print(f"Loaded {name}: {path}")
    else:
        print(f"Missing {name}: {path}")

oof_scores = loaded.get("oof_scores", None)
tuned_models = loaded.get("tuned_models", None)

print("\nCurrent artifact status:")
print("oof_scores :", oof_scores is not None)
print("tuned_models:", tuned_models is not None)

Loaded oof_scores: phase1_artifacts\oof_scores.joblib
Loaded tuned_models: phase1_artifacts\tuned_models.joblib
Loaded saved_y_train: phase1_artifacts\y_train.joblib

Current artifact status:
oof_scores : True
tuned_models: True


## 6. Verify the loaded Phase-1 objects

Expected models:

- Random Forest
- XGBoost
- LightGBM
- CatBoost
- SVM

The SVM in the current project is `LinearSVC`.

In [10]:
MODEL_NAMES = [
    "Random Forest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "SVM"
]

if tuned_models is not None:
    print("Tuned models available:")
    for name in MODEL_NAMES:
        print(f"  {name}: {name in tuned_models}")

if oof_scores is not None:
    print("\nOOF predictions available:")
    for name in MODEL_NAMES:
        print(f"  {name}: {name in oof_scores}")

Tuned models available:
  Random Forest: True
  XGBoost: True
  LightGBM: True
  CatBoost: True
  SVM: True

OOF predictions available:
  Random Forest: True
  XGBoost: True
  LightGBM: True
  CatBoost: True
  SVM: True


## 7. Optional OOF regeneration

If `oof_scores.joblib` is missing but `tuned_models.joblib` exists, this section can regenerate OOF predictions.

**Important:** this does not perform hyperparameter tuning.

It clones each already-tuned model and fits it on the training folds to produce genuine out-of-fold predictions.

If OOF predictions already exist, this section is skipped.

In [11]:
def get_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


if oof_scores is None and tuned_models is not None:
    print("OOF predictions are missing.")
    print("Regenerating OOF predictions from the existing tuned models...")

    oof_scores = {
        name: np.zeros(len(X_selected), dtype=float)
        for name in MODEL_NAMES
        if name in tuned_models
    }

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    for name in MODEL_NAMES:
        if name not in tuned_models:
            continue

        print(f"\nGenerating OOF predictions: {name}")

        for fold, (train_idx, val_idx) in enumerate(
            skf.split(X_selected, y_train), start=1
        ):
            model = clone(tuned_models[name])

            X_tr = X_selected.iloc[train_idx]
            X_val = X_selected.iloc[val_idx]
            y_tr = y_train.iloc[train_idx]

            model.fit(X_tr, y_tr)

            oof_scores[name][val_idx] = get_score(model, X_val)

            print(f"  Fold {fold}/{N_SPLITS} complete")

    print("\nOOF regeneration complete.")

elif oof_scores is not None:
    print("OOF predictions already exist. No regeneration required.")

else:
    print(
        "Cannot generate OOF predictions because tuned_models.joblib "
        "is also missing."
    )

OOF predictions already exist. No regeneration required.


## 8. Validate OOF predictions

The ensemble notebook requires one OOF score per training sample for each base model.

In [12]:
if oof_scores is not None:
    available_oof = [
        name for name in MODEL_NAMES
        if name in oof_scores
    ]

    print("Available OOF models:", available_oof)

    if len(available_oof) != len(MODEL_NAMES):
        print(
            "WARNING: Some models are missing OOF predictions:",
            [m for m in MODEL_NAMES if m not in available_oof]
        )

    for name in available_oof:
        arr = np.asarray(oof_scores[name])

        if len(arr) != len(y_train):
            raise ValueError(
                f"{name}: OOF length {len(arr)} != "
                f"y_train length {len(y_train)}"
            )

        if not np.isfinite(arr).all():
            raise ValueError(
                f"{name}: OOF predictions contain NaN or infinite values."
            )

    print("OOF validation passed.")

else:
    print("OOF predictions are still unavailable.")

Available OOF models: ['Random Forest', 'XGBoost', 'LightGBM', 'CatBoost', 'SVM']
OOF validation passed.


## 9. Save the complete Phase-1 → Phase-2 artifact package

This is the important handoff.

The ensemble notebook can now load these files without depending on variables remaining in a previous Jupyter kernel.

In [13]:
# Save the selected feature definition
joblib.dump(
    selected_features,
    os.path.join(PHASE1_ARTIFACT_DIR, "selected_features.joblib")
)

# Save the exact training/test feature matrices
joblib.dump(
    X_selected,
    os.path.join(PHASE1_ARTIFACT_DIR, "X_selected.joblib")
)

joblib.dump(
    X_test_selected,
    os.path.join(PHASE1_ARTIFACT_DIR, "X_test_selected.joblib")
)

# Save labels
joblib.dump(
    np.asarray(y_train),
    os.path.join(PHASE1_ARTIFACT_DIR, "y_train.joblib")
)

joblib.dump(
    np.asarray(y_test),
    os.path.join(PHASE1_ARTIFACT_DIR, "y_test.joblib")
)

# Save fitted models if available
if tuned_models is not None:
    joblib.dump(
        tuned_models,
        os.path.join(PHASE1_ARTIFACT_DIR, "tuned_models.joblib")
    )

# Save OOF predictions if available
if oof_scores is not None:
    joblib.dump(
        oof_scores,
        os.path.join(PHASE1_ARTIFACT_DIR, "oof_scores.joblib")
    )

print("Artifacts saved successfully.")

Artifacts saved successfully.


## 10. Final artifact inventory

The expected directory is:

```text
phase1_artifacts/
├── oof_scores.joblib
├── tuned_models.joblib
├── y_train.joblib
├── y_test.joblib
├── X_selected.joblib
├── X_test_selected.joblib
└── selected_features.joblib
```

`oof_scores.joblib` and `tuned_models.joblib` are essential for the full ensemble-development workflow.

The test labels are saved separately so the official test set can be evaluated only after the ensemble is frozen.

In [14]:
print("\n=== FINAL ARTIFACT INVENTORY ===")

for filename in sorted(os.listdir(PHASE1_ARTIFACT_DIR)):
    path = os.path.join(PHASE1_ARTIFACT_DIR, filename)
    size_kb = os.path.getsize(path) / 1024
    print(f"{filename:35s} {size_kb:10.1f} KB")

required_for_ensemble = [
    "oof_scores.joblib",
    "tuned_models.joblib",
    "y_train.joblib",
    "y_test.joblib",
    "X_test_selected.joblib",
    "selected_features.joblib"
]

missing = [
    f for f in required_for_ensemble
    if not os.path.exists(os.path.join(PHASE1_ARTIFACT_DIR, f))
]

print("\nRequired-for-ensemble check:")

if missing:
    print("MISSING:")
    for f in missing:
        print(" -", f)
else:
    print("PASS — all required ensemble artifacts are present.")


=== FINAL ARTIFACT INVENTORY ===
X_selected.joblib                      20035.6 KB
X_test_selected.joblib                  9408.6 KB
oof_scores.joblib                       6849.8 KB
selected_features.joblib                   0.2 KB
tuned_models.joblib                    96455.5 KB
y_test.joblib                            643.5 KB
y_train.joblib                          1370.1 KB

Required-for-ensemble check:
PASS — all required ensemble artifacts are present.


## 11. Final sanity checks

These checks confirm that the artifact package is internally consistent before moving to Phase 2.

In [15]:
assert len(y_train) == len(X_selected), (
    "Training features and labels have different lengths."
)

assert len(y_test) == len(X_test_selected), (
    "Test features and labels have different lengths."
)

assert list(X_selected.columns) == selected_features, (
    "Training selected-feature order is incorrect."
)

assert list(X_test_selected.columns) == selected_features, (
    "Test selected-feature order is incorrect."
)

if oof_scores is not None:
    for name in MODEL_NAMES:
        assert name in oof_scores, f"Missing OOF predictions for {name}"
        assert len(oof_scores[name]) == len(y_train), (
            f"OOF length mismatch for {name}"
        )

print("ALL SANITY CHECKS PASSED.")
print()
print("Phase 1 → Phase 2 handoff is ready.")

ALL SANITY CHECKS PASSED.

Phase 1 → Phase 2 handoff is ready.
